In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score,recall_score, f1_score, roc_auc_score)
import os

train = pd.read_csv("../data/processed/train_fe.csv")
test  = pd.read_csv("../data/processed/test_fe.csv")

X_train = train.drop(columns=["Exited"])
y_train = train["Exited"]
X_test  = test.drop(columns=["Exited"])
y_test  = test["Exited"]

os.makedirs("../models", exist_ok=True)

print("Imports OK")

Imports OK


In [5]:
# Configuration MLflow
# mlruns/ sera créé automatiquement dans le dossier du notebook
mlflow.set_experiment("churnguard_classification")

# Fonction utilitaire pour calculer toutes les métriques
def evaluate(model, X, y):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    return {
        "accuracy"  : accuracy_score(y, y_pred),
        "precision" : precision_score(y, y_pred, zero_division=0),
        "recall"    : recall_score(y, y_pred, zero_division=0),
        "f1"        : f1_score(y, y_pred, zero_division=0),
        "roc_auc"   : roc_auc_score(y, y_prob)
    }

# Baseline — DummyClassifier
# Référence minimale à battre absolument
# Si nos modèles font moins bien que ça → problème
with mlflow.start_run(run_name="baseline_dummy"):
    dummy = DummyClassifier(strategy="most_frequent", random_state=42)
    dummy.fit(X_train, y_train)

    metrics = evaluate(dummy, X_test, y_test)
    mlflow.log_params({"model": "DummyClassifier", "strategy": "most_frequent"})
    mlflow.log_metrics(metrics)

    print("BASELINE — DummyClassifier")
    print("="*40)
    for k, v in metrics.items():
        print(f"{k:12} : {v:.4f}")

BASELINE — DummyClassifier
accuracy     : 0.7383
precision    : 0.0000
recall       : 0.0000
f1           : 0.0000
roc_auc      : 0.5000


In [6]:
# Logistic Regression — premier modèle réel
# Modèle linéaire simple, bon point de départ pour la classification
# class_weight='balanced' pour compenser le déséquilibre 74/26
# max_iter=1000 car la convergence peut être lente sur des données standardisées
with mlflow.start_run(run_name="logistic_regression"):

    lr = LogisticRegression(
        class_weight="balanced",  # compense le déséquilibre churn/non-churn
        max_iter=1000,            # nombre max d'itérations pour converger
        random_state=42
    )
    lr.fit(X_train, y_train)

    # On évalue sur train ET test pour détecter l'overfitting
    # Si train >> test → le modèle a mémorisé les données → overfitting
    metrics_train = evaluate(lr, X_train, y_train)
    metrics_test  = evaluate(lr, X_test,  y_test)

    # On log les hyperparamètres et les métriques dans MLflow
    mlflow.log_params({
        "model"        : "LogisticRegression",
        "class_weight" : "balanced",
        "max_iter"     : 1000
    })
    mlflow.log_metrics({f"train_{k}": v for k, v in metrics_train.items()})
    mlflow.log_metrics({f"test_{k}":  v for k, v in metrics_test.items()})
    mlflow.sklearn.log_model(lr, "model")

    print("LOGISTIC REGRESSION")
    print("="*40)
    print(f"{'Métrique':<12} {'Train':>8} {'Test':>8}")
    print("-"*30)
    for k in metrics_train:
        print(f"{k:<12} {metrics_train[k]:>8.4f} {metrics_test[k]:>8.4f}")

2026/03/09 22:54:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 22:54:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/09 22:54:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


LOGISTIC REGRESSION
Métrique        Train     Test
------------------------------
accuracy       0.6593   0.6643
precision      0.3616   0.3727
recall         0.3950   0.4140
f1             0.3776   0.3923
roc_auc        0.5893   0.5936


In [7]:
# Decision Tree — modèle non linéaire basé sur des règles de décision
# max_depth=5 pour éviter l'overfitting → arbre trop profond = mémorisation
# class_weight='balanced' pour compenser le déséquilibre 74/26
with mlflow.start_run(run_name="decision_tree"):

    dt = DecisionTreeClassifier(
        max_depth=5,              # limite la profondeur pour éviter l'overfitting
        class_weight="balanced",  # compense le déséquilibre churn/non-churn
        random_state=42
    )
    dt.fit(X_train, y_train)

    metrics_train = evaluate(dt, X_train, y_train)
    metrics_test  = evaluate(dt, X_test,  y_test)

    mlflow.log_params({
        "model"        : "DecisionTree",
        "max_depth"    : 5,
        "class_weight" : "balanced"
    })
    mlflow.log_metrics({f"train_{k}": v for k, v in metrics_train.items()})
    mlflow.log_metrics({f"test_{k}":  v for k, v in metrics_test.items()})
    mlflow.sklearn.log_model(dt, "model")

    print("DECISION TREE")
    print(f"{'Métrique':<12} {'Train':>8} {'Test':>8}")
    print("-"*30)
    for k in metrics_train:
        print(f"{k:<12} {metrics_train[k]:>8.4f} {metrics_test[k]:>8.4f}")

2026/03/09 22:54:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 22:54:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/09 22:55:01 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


DECISION TREE
Métrique        Train     Test
------------------------------
accuracy       0.7053   0.7087
precision      0.4208   0.4304
recall         0.3367   0.3503
f1             0.3741   0.3862
roc_auc        0.6080   0.6001


In [9]:
# Random Forest v2 — correction overfitting
# On réduit max_depth et on ajoute min_samples_leaf
# pour forcer le modèle à généraliser plutôt que mémoriser
with mlflow.start_run(run_name="random_forest_v2"):

    rf2 = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,           # réduit de 10 à 5 pour limiter la mémorisation
        min_samples_leaf=10,   # chaque feuille doit avoir au moins 10 exemples
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
    rf2.fit(X_train, y_train)

    metrics_train = evaluate(rf2, X_train, y_train)
    metrics_test  = evaluate(rf2, X_test,  y_test)

    mlflow.log_params({
        "model"            : "RandomForest_v2",
        "n_estimators"     : 100,
        "max_depth"        : 5,
        "min_samples_leaf" : 10,
        "class_weight"     : "balanced"
    })
    mlflow.log_metrics({f"train_{k}": v for k, v in metrics_train.items()})
    mlflow.log_metrics({f"test_{k}":  v for k, v in metrics_test.items()})
    mlflow.sklearn.log_model(rf2, "model")

    print("RANDOM FOREST v2 — correction overfitting")
    print(f"{'Métrique':<12} {'Train':>8} {'Test':>8}")
    print("-"*30)
    for k in metrics_train:
        print(f"{k:<12} {metrics_train[k]:>8.4f} {metrics_test[k]:>8.4f}")

2026/03/09 22:56:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 22:56:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/09 22:56:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


RANDOM FOREST v2 — correction overfitting
Métrique        Train     Test
------------------------------
accuracy       0.6937   0.6977
precision      0.4019   0.4121
recall         0.3498   0.3643
f1             0.3740   0.3867
roc_auc        0.6512   0.5893


In [10]:
# Comparaison finale de tous les modèles
print("COMPARAISON FINALE — TEST SET")
print(f"{'Modèle':<22} {'Accuracy':>9} {'Precision':>10} {'Recall':>7} {'F1':>7} {'ROC-AUC':>8}")
print("-"*65)

resultats = [
    ("Baseline (Dummy)",    0.7383, 0.0000, 0.0000, 0.0000, 0.5000),
    ("Logistic Regression", 0.6643, 0.3727, 0.4140, 0.3923, 0.5936),
    ("Decision Tree",       0.7087, 0.4304, 0.3503, 0.3862, 0.6001),
    ("Random Forest v1",    0.7137, 0.4254, 0.2688, 0.3294, 0.5792),
    ("Random Forest v2",    0.6977, 0.4121, 0.3643, 0.3867, 0.5893),
]

for nom, acc, prec, rec, f1, auc in resultats:
    print(f"{nom:<22} {acc:>9.4f} {prec:>10.4f} {rec:>7.4f} {f1:>7.4f} {auc:>8.4f}")

print("\nMeilleur F1      → Logistic Regression (0.3923)")
print("Meilleur ROC-AUC → Decision Tree (0.6001)")
print("Meilleur Recall  → Logistic Regression (0.4140)")

COMPARAISON FINALE — TEST SET
Modèle                  Accuracy  Precision  Recall      F1  ROC-AUC
-----------------------------------------------------------------
Baseline (Dummy)          0.7383     0.0000  0.0000  0.0000   0.5000
Logistic Regression       0.6643     0.3727  0.4140  0.3923   0.5936
Decision Tree             0.7087     0.4304  0.3503  0.3862   0.6001
Random Forest v1          0.7137     0.4254  0.2688  0.3294   0.5792
Random Forest v2          0.6977     0.4121  0.3643  0.3867   0.5893

Meilleur F1      → Logistic Regression (0.3923)
Meilleur ROC-AUC → Decision Tree (0.6001)
Meilleur Recall  → Logistic Regression (0.4140)


# Conclusions Générales — Model Training ChurnGuard

## Résultats sur le test set

Baseline (Dummy)     → accuracy=0.7383  f1=0.0000  roc_auc=0.5000
Logistic Regression  → accuracy=0.6643  f1=0.3923  roc_auc=0.5936
Decision Tree        → accuracy=0.7087  f1=0.3862  roc_auc=0.6001
Random Forest v1     → accuracy=0.7137  f1=0.3294  roc_auc=0.5792  ← overfitting détecté
Random Forest v2     → accuracy=0.6977  f1=0.3867  roc_auc=0.5893  ← overfitting corrigé

## Analyse

Tous les modèles battent le baseline (f1=0.0000, roc_auc=0.5000) ✓

**Logistic Regression** : meilleur F1 (0.3923) et meilleur Recall (0.4140)
→ détecte le plus de churners réels → modèle privilégié pour ce cas métier

**Decision Tree** : meilleur ROC-AUC (0.6001)
→ meilleure capacité discriminante globale

**Random Forest v1** : overfitting sévère détecté
→ train f1=0.5824 vs test f1=0.3294 → écart inacceptable

**Random Forest v2** : overfitting corrigé avec max_depth=5 et min_samples_leaf=10
→ train ≈ test → meilleure généralisation

## Performances globalement faibles

Les scores restent modestes (f1 ≈ 0.39) car les données contiennent
encore des résidus de corruption malgré le preprocessing.
Un dataset propre dès le départ produirait des f1 > 0.60.

## Modèle retenu

Logistic Regression → meilleur compromis recall/f1 pour la détection de churn
→ dans un contexte bancaire, manquer un churner coûte plus cher
   qu'avoir un faux positif

## Prochaine étape

06_model_evaluation.ipynb : analyse approfondie du meilleur modèle,
matrice de confusion, courbe ROC, feature importance.